# RFRec — H&M Sampled Dataset

Regularized Federated Recommendation: each client holds private user embeddings
and a local copy of item embeddings. A server aggregates item embeddings each
round and penalises client drift via an L2 regularisation term.


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools
from sklearn.model_selection import train_test_split


## 1. Load Data


In [4]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)

n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users : {n_users}")
print(f"Total Items : {n_items}")
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
train_df.head()


Total Users : 1760
Total Items : 2881
Train : 62568 | Val : 15643 | Test : 19553


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


## 2. Build Rating Matrices


In [6]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["user_id"].values, dtype=torch.long)
    items = torch.tensor(df["item_id"].values, dtype=torch.long)
    mask  = (users >= 0) & (users < n_u) & (items >= 0) & (items < n_i)
    if (~mask).any():
        print(f"  [make_matrix] dropping {(~mask).sum().item()} out-of-bounds rows")
    users, items = users[mask], items[mask]
    denom = max(max_r - min_r, 1e-8)
    vals  = torch.tensor(
        (df["bought"].values[mask.numpy()] - min_r) / denom, dtype=torch.float32
    )
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)

print(f"Train : {(train_matrix != -1).sum().item()} observed")
print(f"Val   : {(val_matrix   != -1).sum().item()} observed")
print(f"Test  : {(test_matrix  != -1).sum().item()} observed")


Train : 61386 observed
Val   : 15572 observed
Test  : 19450 observed


## 3. Loss


In [8]:
class RFRecLoss(nn.Module):
    """MSE + L2 user reg + L2 item reg + drift penalty (lam_L * ||v - aver_v||^2)."""
    def __init__(self, lam_u=0.1, lam_v=0.0, lam_L=10.0):
        super().__init__()
        self.lam_u = lam_u
        self.lam_v = lam_v
        self.lam_L = lam_L

    def forward(self, matrix, u_features, v_features, aver_v):
        mask      = (matrix != -1).float()
        predicted = torch.sigmoid(torch.mm(u_features, v_features.t()))
        pred_err  = torch.sum(((matrix - predicted) ** 2) * mask)
        u_reg     = self.lam_u * torch.sum(u_features ** 2)
        v_reg     = self.lam_v * torch.sum(v_features ** 2)
        drift     = self.lam_L * torch.sum((v_features - aver_v) ** 2)
        return pred_err + u_reg + v_reg + drift, pred_err


def compute_rmse(matrix, predicted, min_r, max_r):
    mask    = (matrix != -1).float()
    n       = mask.sum()
    if n == 0: return float('nan')
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix    * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())

print("RFRecLoss, compute_rmse defined.")


RFRecLoss, compute_rmse defined.


## 4. Parameter Tuning -- Grid Search

Searches `lr`, `lam_u`, `latent_vectors` (lam_L fixed at 10, lam_v=0 as in paper).
Run **once** after building matrices.


In [10]:
LR_GRID     = [0.001, 0.01, 0.1]
LAM_GRID    = [0.01,  0.1,  0.3]
LATENT_GRID = [10,    20,   40 ]
TUNE_EPOCHS = 20
TUNE_PAT    = 5
TUNE_N_USERS = min(50, n_users)   # subset for speed

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)
grid_results = []
tune_train = train_matrix[:TUNE_N_USERS]
tune_val   = val_matrix[:TUNE_N_USERS]
NC_T = TUNE_N_USERS   # 1 client per user during tuning

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)
    _ufs = [torch.randn(1, _K, requires_grad=True) for _ in range(NC_T)]
    _vfs = [torch.randn(n_items, _K, requires_grad=True) for _ in range(NC_T)]
    for _uf, _vf in zip(_ufs, _vfs):
        _uf.data.mul_(0.01); _vf.data.mul_(0.01)
    _loss_fn = RFRecLoss(lam_u=_lam, lam_v=0.0, lam_L=10.0)
    _opts    = [torch.optim.Adam([_ufs[i], _vfs[i]], lr=_lr) for i in range(NC_T)]
    _aver    = sum(_vfs[i].data for i in range(NC_T)) / NC_T
    _best_val, _wait = float('inf'), 0
    for _ep in range(TUNE_EPOCHS):
        _tmp = torch.zeros(n_items, _K)
        for i in range(NC_T):
            _opts[i].zero_grad()
            _loss, _ = _loss_fn(tune_train[i:i+1], _ufs[i], _vfs[i], _aver)
            _loss.backward(retain_graph=True)
            _opts[i].step()
            _tmp += _vfs[i].data
        _aver = _tmp / NC_T
        with torch.no_grad():
            _pred = torch.cat(
                [torch.sigmoid(torch.mm(_ufs[i], _aver.t())) for i in range(NC_T)], dim=0)
            _v = compute_rmse(tune_val, _pred, min_rating, max_rating)
        if _v < _best_val: _best_val, _wait = _v, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT: break
    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_val:>14.4f}{_marker}")
best_val_t, best_lr, best_lam, best_K = min(grid_results, key=lambda x: x[0])
print(f"\nBest val RMSE  : {best_val_t:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell.")


Grid search: 27 combinations x 20 epochs
  #       lr     lam     K   best val RMSE
------------------------------------------
  1   0.0010   0.010    10         11.0352 <--
  2   0.0010   0.010    20         11.0352 <--
  3   0.0010   0.010    40         11.0352 <--
  4   0.0010   0.100    10         11.0352
  5   0.0010   0.100    20         11.0352 <--
  6   0.0010   0.100    40         11.0352
  7   0.0010   0.300    10         11.0352
  8   0.0010   0.300    20         11.0352 <--
  9   0.0010   0.300    40         11.0352
 10   0.0100   0.010    10         11.0350 <--
 11   0.0100   0.010    20         11.0347 <--
 12   0.0100   0.010    40         11.0346 <--
 13   0.0100   0.100    10         11.0351
 14   0.0100   0.100    20         11.0351
 15   0.0100   0.100    40         11.0352
 16   0.0100   0.300    10         11.0352
 17   0.0100   0.300    20         11.0352
 18   0.0100   0.300    40         11.0352
 19   0.1000   0.010    10         11.0234 <--
 20   0.1000   0.010

## 5. Training


In [12]:
try: latent_vectors
except NameError: latent_vectors = 20
try: lr
except NameError: lr = 0.1
try: lam
except NameError: lam = 0.1
num_epoch  = 100
patience   = 10
NUM_CLIENTS = n_users   # 1 client per user
lam_L      = 10.0

torch.manual_seed(0)
user_features = [torch.randn(1, latent_vectors, requires_grad=True) for _ in range(NUM_CLIENTS)]
item_features = [torch.randn(n_items, latent_vectors, requires_grad=True) for _ in range(NUM_CLIENTS)]
for i in range(NUM_CLIENTS):
    user_features[i].data.mul_(0.01)
    item_features[i].data.mul_(0.01)
aver_item = sum(item_features[i].data for i in range(NUM_CLIENTS)) / NUM_CLIENTS

loss_fn   = RFRecLoss(lam_u=lam, lam_v=0.0, lam_L=lam_L)
optimizers = [torch.optim.Adam([user_features[i], item_features[i]], lr=lr)
              for i in range(NUM_CLIENTS)]

best_val_rmse, patience_count = float('inf'), 0
best_uf, best_aver = None, None

for epoch in range(num_epoch):
    tmp = torch.zeros(n_items, latent_vectors)
    for i in range(NUM_CLIENTS):
        optimizers[i].zero_grad()
        loss, _ = loss_fn(train_matrix[i:i+1], user_features[i], item_features[i], aver_item)
        loss.backward(retain_graph=True)
        optimizers[i].step()
        tmp += item_features[i].data
    aver_item = tmp / NUM_CLIENTS

    with torch.no_grad():
        pred_parts = [torch.sigmoid(torch.mm(user_features[i], aver_item.t())) for i in range(NUM_CLIENTS)]
        pred_full  = torch.cat(pred_parts, dim=0)
        train_rmse = compute_rmse(train_matrix, pred_full, min_rating, max_rating)
        val_rmse   = compute_rmse(val_matrix,   pred_full, min_rating, max_rating)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train RMSE: {train_rmse:.4f} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_uf   = [user_features[i].data.clone() for i in range(NUM_CLIENTS)]
        best_aver = aver_item.clone()
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

with torch.no_grad():
    for i in range(NUM_CLIENTS): user_features[i].data.copy_(best_uf[i])
    aver_item = best_aver
print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Epoch   0 | train RMSE: 11.1025 | val RMSE: 11.0902
Epoch  10 | train RMSE: 11.1022 | val RMSE: 11.0902
Epoch  20 | train RMSE: 11.0995 | val RMSE: 11.0901
Epoch  30 | train RMSE: 11.0857 | val RMSE: 11.0892
Epoch  40 | train RMSE: 11.0410 | val RMSE: 11.0840
Epoch  50 | train RMSE: 10.9352 | val RMSE: 11.0619
Epoch  60 | train RMSE: 10.7344 | val RMSE: 10.9920
Epoch  70 | train RMSE: 10.4008 | val RMSE: 10.8162
Epoch  80 | train RMSE: 9.8846 | val RMSE: 10.4534
Epoch  90 | train RMSE: 9.1423 | val RMSE: 9.8335

Best val RMSE: 9.0624


## 6. Test Evaluation


In [14]:
with torch.no_grad():
    pred_parts = [torch.sigmoid(torch.mm(user_features[i], aver_item.t())) for i in range(NUM_CLIENTS)]
    pred_test  = torch.cat(pred_parts, dim=0)
    test_rmse  = compute_rmse(test_matrix, pred_test, min_rating, max_rating)
    mask     = (test_matrix != -1).float()
    n_obs    = mask.sum()
    pred_sc  = pred_test   * (max_rating - min_rating) + min_rating
    true_sc  = test_matrix * (max_rating - min_rating) + min_rating
    test_mae = (torch.abs(pred_sc - true_sc) * mask).sum() / n_obs
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae.item():.4f}")


Test RMSE : 9.0516
Test MAE  : 8.5570
